In [ ]:
import pulp as pl


# ----- SETS (MENGEN) & PARAMETER --------#

# --- SETS (Mengen) ---
V = [row['type'] for _, row in all_data['vehicles'].iterrows()] 
# Menge der verfügbaren Fahrzeuge  => ['MAN_eTGX', 'Volvo_FH_Electric', 'Mercedes_eActros_600']

L = [row['customer_id'] for _, row in all_data['delivery_routes'].iterrows()]
# [1, 2, 3, 4, 5]
# L =  all_data['delivery_routes']             
# Lieferungen (Standort(lat und long), un-Nummer,substanzname, Klasse, quantity, unit, packaging_type )

K = [all_data['delivery_routes'].iloc[i]['danger_class'] for i in range(len(all_data['delivery_routes']))]         
# Gefahrgutklassen der einzelnen Lieferungen =>['3', '2 (TOC)', '3', '1.1D', '3']

# Richtiges Format für den Solver: [172539, 172541, 172542, ]
N = df_nodes['node'].tolist()
# Alle Knoten des Straßennetzes

# Richtiges Format für den Solver: [(1822620447, 11405216193), (11405216193, 2845478638), ]
E = list(zip(df_edges['u'], df_edges['v']))
# Gerichtete Kanten im Straßennetz 


# --- PARAMETER (Eingabedaten) ---

# Lieferungsdaten (Bezug zu L)
Dem = {row['customer_id']: row['quantity'] for _, row in all_data['delivery_routes'].iterrows()}                
# Gewicht je Lieferung (kg) => Annahme das 1 L = 1 Kg ist 
#{1: 15000, 2: 5000, 3: 21500, 4: 1500, 5: 1000, 6: 19500, 7: 15000, 8: 10000, 9: 2500, 10: 8000}}

Class = {row['customer_id']: row['danger_class'] for _, row in all_data['delivery_routes'].iterrows()} 
# Gefahrgutklasse je Lieferung
#{1: '3', 2: '2 (TOC)', 3: '3', 4: '1.1D', 5: '3', 6: '8', 7: '9', 8: '2', 9: '3', 10: '9'}


depot_node = nearest_node_result["nearest_node"]
lieferungen = all_data['customer_locations']["Nr"]
O_dep = {
    lieferung_id: int(depot_node)
    for lieferung_id in lieferungen
} 

D_dep = {
    customer_id: info["nearest_node"]
    for customer_id, info in customer_to_node.items()
}     # Zielknoten (Destination)            

# Fahrzeugdaten (Bezug zu V) aus (vehicles.csv)
Cap = {row['type']: row['capacity_kg'] for _, row in all_data['vehicles'].iterrows()}    
# Maximale Nutzlast (t) => {'MAN_eTGX': 18, 'Volvo_FH_Electric': 20, 'Mercedes_eActros_600': 22}  

Range = {row['type']: row['range_km'] for _, row in all_data['vehicles'].iterrows()}    
# Akkureichweite => {'MAN_eTGX': 500, 'Volvo_FH_Electric': 470, 'Mercedes_eActros_600': 500} 

FC = {row['type']: row['fixed_cost'] for _, row in all_data['vehicles'].iterrows()}    
# Fixkosten (€/Tag) => {'MAN_eTGX': 180, 'Volvo_FH_Electric': 200, 'Mercedes_eActros_600': 220}

# --- Netzwerk- & Risikodaten (Bezug zu E und K) ---

Dist = {
    (row["u"], row["v"]): float(row["distance"]) / 1000
    for _, row in df_edges.iterrows()
}
# Länge der Kante in km              

#Risiko muss hier berechnet werden mit parametern alpha, beta und gamma
Risk = Risk_calc          # Risikoscore [0, 1] je Kante und Klasse       HIER NOCH MACHEN 
Allow = Allow_calc            # 1 wenn erlaubt, 0 wenn gesperrt (ADR)        HIER NOCH MACHEN 
# Beispiel für eine Sperre: Allow[(('n1', 'ziel_A'), 'klasse_6')] = 0

# Fahrtkosten (Bezug zu V und E)
VC_per_km = all_data['vehicles'].set_index('type')['variable_cost_per_km'].to_dict()       #variable Kosten pro km je Fahrzeugtyp
Energy_per_km = all_data['vehicles'].set_index('type')['energy_kwh_per_km'].to_dict()      # Energieverbrauch pro km je Fahrzeugtyp (kWh/km)

strompreis = 0.35
# Strompreis erstmal fest lassen, weil wir nicht wissen ob Straßentyp an kanten dran steht 
VC = {}
for v in V:
    for e in E:
        # Dist[e] muss vorher definiert sein (z.B. als Dictionary Dist = {e: länge_in_km})
        distanz = Dist[e]
        
        # km-Kosten berechnen
        km_kosten = distanz * VC_per_km[v]
        
        # Energiekosten berechnen
        energie_kosten = distanz * Energy_per_km[v] * strompreis
        
        # Gesamte variable Kosten für dieses Fahrzeug auf dieser Kante
        VC[(v, e)] = km_kosten + energie_kosten

      # Variable Kosten (€) pro Fahrzeug und Kante

# Gewichtungsfaktoren für die Zielfunktion
w1 = 0.5  # Risiko-Gewicht
w2 = 0.5  # Kosten-Gewicht



In [ ]:
#----------------------  MODELL INITIALISIERUNG--------------------------#
model = pl.LpProblem("Gefahrgut_Routenplanung_MILP", pl.LpMinimize)



#---------------------- ENTSCHEIDUNGSVARIABLEN--------------------------#

# f[l, e]: Binär - 1, wenn Lieferung l über Kante e transportiert wird
f = pl.LpVariable.dicts("Fluss", [(l, e) for l in L for e in E], cat='Binary')

# y[l, v]: Binär - 1, wenn Lieferung l dem Fahrzeug v zugewiesen wird
y = pl.LpVariable.dicts("Zuweisung", [(l, v) for l in L for v in V], cat='Binary')

# z[v]: Binär - 1, wenn Fahrzeug v heute eingesetzt wird (Aktivierung)
z = pl.LpVariable.dicts("Aktivierung", [v for v in V], cat='Binary')

# u[l, n]: Stetig - MTZ-Hilfsvariable für die Besuchsreihenfolge (Subtour-Eliminierung)
u = pl.LpVariable.dicts("MTZ", [(l, n) for l in L for n in N], lowBound=0, upBound=len(N), cat='Continuous')



#----------------  ZIELFUNKTION (Normiert & Gewichtet)-------------------#

# Berechnung der theoretischen Maxima für Skalierung [0, 1]
Risk_max = max(Risk[(e, k)] for e in E for k in K)
Risk_sum_max = Risk_max * len(L) * len(E) 

Cost_max = (sum(FC[v] for v in V) 
          + sum(max(VC[(v, e)] for v in V) for e in E) * len(L))

# Einzelterme aufbauen
risk_term = pl.lpSum(Risk[(e, Class[l])] * f[(l, e)] for l in L for e in E)
fixed_cost_term = pl.lpSum(FC[v] * z[v] for v in V)
variable_cost_term = pl.lpSum(
    VC[(v, e)] * y[(l, v)] * (1 / len(V)) 
    for l in L for v in V for e in E 
    if Allow.get((e, Class[l]), 1) == 1
)

# Dem Modell hinzufügen
model += (
    (w1 / Risk_sum_max) * risk_term + 
    (w2 / Cost_max) * (fixed_cost_term + variable_cost_term)
), "Minimiere_Risiko_und_Kosten_normiert"



#--------------- NEBENBEDINGUNGEN (Constraints) --------------------------#


# --- C1: Transportpflicht ---
# Jede Lieferung muss exakt einem Fahrzeug zugewiesen werden.
for l in L:
    model += pl.lpSum(y[(l, v)] for v in V) == 1, f"C1_Transportpflicht_{l}"

# --- C2: Kapazitätsbedingung ---
# Gesamtgewicht der zugewiesenen Lieferungen darf die LKW-Nutzlast nicht überschreiten.
for v in V:
    model += pl.lpSum(Dem[l] * y[(l, v)] for l in L) <= Cap[v] * z[v], f"C2_Kapazitaet_{v}"

# --- C3: Aktivierungsbedingung ---
# Fahrzeug v muss als aktiv markiert sein (z[v]=1), wenn ihm eine Lieferung zugewiesen wird.
for l in L:
    for v in V:
        model += z[v] >= y[(l, v)], f"C3_Aktivierung_{l}_{v}"

# --- C4: Gefahrgutrestriktion ---
# Eine Kante darf von einer Lieferung nicht befahren werden, wenn sie gesperrt ist.
# (Es werden aus Performance-Gründen nur Constraints für gesperrte Kanten erzeugt).
for l in L:
    for e in E:
        if Allow.get((e, Class[l]), 1) == 0:
            model += f[(l, e)] <= 0, f"C4_Gefahrgut_Sperre_{l}_{e[0]}_{e[1]}"

# --- C5: Flusserhaltung (Network Flow Conservation) ---
# Garantiert eine zusammenhängende Route vom Start- zum Zielknoten.
in_edges = {n: [e for e in E if e[1] == n] for n in N}
out_edges = {n: [e for e in E if e[0] == n] for n in N}

for l in L:
    for n in N:
        inflow = pl.lpSum(f[(l, e)] for e in in_edges[n])
        outflow = pl.lpSum(f[(l, e)] for e in out_edges[n])

        if n == O_dep[l]:      # Startknoten (Quelle)
            model += outflow - inflow == 1, f"C5_Flow_Start_{l}_{n}"
        elif n == D_dep[l]:    # Zielknoten (Senke)
            model += inflow - outflow == 1, f"C5_Flow_Ziel_{l}_{n}"
        else:                  # Durchgangsknoten
            model += inflow == outflow, f"C5_Flow_Trans_{l}_{n}"

# --- C6: Subtour-Eliminierung (Miller-Tucker-Zemlin) ---
# Verhindert, dass der Solver isolierte, physikalisch unvollständige Kreise (Schleifen) bildet.
M_mtz = len(N)  # Engste mathematisch korrekte Schranke
for l in L:
    for (i, j) in E:
        if j != O_dep[l]:  # Startknoten hat keinen logischen Vorgänger auf der Route
            model += u[(l, i)] - u[(l, j)] + M_mtz * f[(l, (i, j))] <= M_mtz - 1, f"C6_MTZ_{l}_{i}_{j}"

# --- C7: Akkureichweite (E-Truck spezifisch) ---
# Gesamtdistanz der geplanten Route darf die Reichweite des zugewiesenen LKWs nicht überschreiten.
for l in L:
    total_dist = pl.lpSum(Dist[e] * f[(l, e)] for e in E)
    max_range = pl.lpSum(Range[v] * y[(l, v)] for v in V)
    model += total_dist <= max_range, f"C7_Reichweite_{l}"



# --------------------- MODELL LÖSEN --------------------------------#

solver = pl.PULP_CBC_CMD(msg=1, timeLimit=60)
status = model.solve(solver)
print(f"Status: {pl.LpStatus[status]}")